In [35]:
import os
import glob
import gc
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

In [36]:
# =========================================================
# CONFIG
# =========================================================
DATA_DIR = r"C:\Users\abdal\OneDrive\Desktop\chess-project\pro moves"          # folder containing new .npz files
CHECKPOINT_PATH = r"C:\Users\abdal\OneDrive\Desktop\chess-project\ver1.2\last_model.pth"   # old trained checkpoint
SAVE_DIR = r"C:\Users\abdal\OneDrive\Desktop\chess-project\ver1.3"

BATCH_SIZE = 256
NUM_EPOCHS = 3
LEARNING_RATE = 1e-4      # smaller LR is usually safer for retraining
WEIGHT_DECAY = 1e-5
TRAIN_VAL_SPLIT = 0.9
NUM_WORKERS = 0           # for Windows/local safer to keep 0 first
SEED = 42

INPUT_CHANNELS = 12
NUM_BLOCKS = 12
CHANNELS = 128
POLICY_SIZE = 4864
VALUE_HIDDEN = 256
VALUE_DROPOUT = 0.15

os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Using device: cuda


In [37]:
# =========================================================
# DATASET FOR ONE NPZ FILE
# =========================================================
class SingleNPZDataset(Dataset):
    def __init__(self, file_path):
        data = np.load(file_path)

        self.features = data["features"].astype(np.float32)
        self.policy_labels = data["policy_labels"].astype(np.int64)
        self.value_labels = data["value_labels"].astype(np.float32)

        assert self.features.shape[1:] == (12, 8, 8), f"Expected (12,8,8), got {self.features.shape[1:]}"
        assert len(self.features) == len(self.policy_labels) == len(self.value_labels)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        p = torch.tensor(self.policy_labels[idx], dtype=torch.long)
        v = torch.tensor(self.value_labels[idx], dtype=torch.float32)
        return x, p, v


In [38]:
class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x):
        # x: (B, C, H, W)
        b, c, _, _ = x.size()
        s = F.adaptive_avg_pool2d(x, 1).view(b, c)
        s = F.relu(self.fc1(s), inplace=True)
        s = torch.sigmoid(self.fc2(s)).view(b, c, 1, 1)
        return x * s


class SEResidualBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)

        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

        self.se = SEBlock(channels, reduction=reduction)

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out, inplace=True)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)

        out = out + residual
        out = F.relu(out, inplace=True)
        return out


class AlphaZeroSEResNet(nn.Module):
    def __init__(
        self,
        input_channels: int = 12,
        num_blocks: int = 12,
        channels: int = 128,
        policy_size: int = 4864,
        value_hidden: int = 256,
        value_dropout: float = 0.15,
    ):
        super().__init__()

        # Stem
        self.input_conv = nn.Conv2d(input_channels, channels, kernel_size=3, padding=1, bias=False)
        self.input_bn = nn.BatchNorm2d(channels)

        # Residual tower
        self.res_tower = nn.Sequential(
            *[SEResidualBlock(channels, reduction=16) for _ in range(num_blocks)]
        )

        # Policy head
        self.policy_conv = nn.Conv2d(channels, 32, kernel_size=1, bias=False)
        self.policy_bn = nn.BatchNorm2d(32)
        self.policy_fc = nn.Linear(32 * 8 * 8, policy_size)

        # Value head
        self.value_conv = nn.Conv2d(channels, 8, kernel_size=1, bias=False)
        self.value_bn = nn.BatchNorm2d(8)
        self.value_fc1 = nn.Linear(8 * 8 * 8, value_hidden)
        self.value_dropout = nn.Dropout(value_dropout)
        self.value_fc2 = nn.Linear(value_hidden, 1)

    def forward(self, x):
        x = self.input_conv(x)
        x = self.input_bn(x)
        x = F.relu(x, inplace=True)

        x = self.res_tower(x)

        # Policy
        p = self.policy_conv(x)
        p = self.policy_bn(p)
        p = F.relu(p, inplace=True)
        p = p.view(p.size(0), -1)
        policy_logits = self.policy_fc(p)

        # Value
        v = self.value_conv(x)
        v = self.value_bn(v)
        v = F.relu(v, inplace=True)
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v), inplace=True)
        v = self.value_dropout(v)
        value = torch.tanh(self.value_fc2(v)).squeeze(1)

        return policy_logits, value

In [39]:
# =========================================================
# METRICS / TRAIN HELPERS
# =========================================================
def policy_accuracy(policy_logits, policy_targets):
    preds = torch.argmax(policy_logits, dim=1)
    return (preds == policy_targets).float().mean().item()


def run_epoch(model, loader, optimizer, policy_criterion, value_criterion, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_policy_loss = 0.0
    total_value_loss = 0.0
    total_acc = 0.0
    total_samples = 0

    for x, p_targets, v_targets in loader:
        x = x.to(DEVICE, non_blocking=True)
        p_targets = p_targets.to(DEVICE, non_blocking=True)
        v_targets = v_targets.to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            policy_logits, value_preds = model(x)

            loss_policy = policy_criterion(policy_logits, p_targets)
            loss_value = value_criterion(value_preds, v_targets)
            loss = loss_policy + loss_value

            if train:
                loss.backward()
                optimizer.step()

        batch_size = x.size(0)
        acc = policy_accuracy(policy_logits, p_targets)

        total_loss += loss.item() * batch_size
        total_policy_loss += loss_policy.item() * batch_size
        total_value_loss += loss_value.item() * batch_size
        total_acc += acc * batch_size
        total_samples += batch_size

    return (
        total_loss / total_samples,
        total_policy_loss / total_samples,
        total_value_loss / total_samples,
        total_acc / total_samples,
    )


In [40]:
# =========================================================
# CHECKPOINT LOADING
# =========================================================
def load_checkpoint(model, optimizer, checkpoint_path):
    if not checkpoint_path or not os.path.isfile(checkpoint_path):
        print("No checkpoint found. Training from scratch.")
        return model, optimizer, 1, float("inf")

    print(f"Loading checkpoint from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"])

    if "optimizer_state_dict" in checkpoint:
        try:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            print("Optimizer state loaded.")
        except Exception as e:
            print(f"Could not load optimizer state: {e}")
            print("Will continue with fresh optimizer state.")

    start_epoch = checkpoint.get("epoch", 0) + 1
    best_val_loss = checkpoint.get("val_loss", float("inf"))

    print(f"Resuming from epoch: {start_epoch}")
    print(f"Previous best val loss: {best_val_loss}")

    return model, optimizer, start_epoch, best_val_loss



In [ ]:
# =========================================================
# MAIN TRAINING LOOP
# =========================================================
def main():
    npz_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.npz")))
    if not npz_files:
        raise FileNotFoundError(f"No .npz files found in {DATA_DIR}")

    print(f"Found {len(npz_files)} npz files.")

    model = AlphaZeroSEResNet(
        input_channels=INPUT_CHANNELS,
        num_blocks=NUM_BLOCKS,
        channels=CHANNELS,
        policy_size=POLICY_SIZE,
        value_hidden=VALUE_HIDDEN,
        value_dropout=VALUE_DROPOUT,
    ).to(DEVICE)

    policy_criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    value_criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    model, optimizer, start_epoch, best_global_val_loss = load_checkpoint(
        model, optimizer, CHECKPOINT_PATH
    )

    end_epoch = start_epoch + NUM_EPOCHS - 1

    for epoch in range(start_epoch, end_epoch + 1):
        print(f"\n{'='*70}")
        print(f"Epoch {epoch}/{end_epoch}")
        print(f"{'='*70}")

        random.shuffle(npz_files)

        epoch_train_loss = 0.0
        epoch_train_pol = 0.0
        epoch_train_val = 0.0
        epoch_train_acc = 0.0

        epoch_val_loss = 0.0
        epoch_val_pol = 0.0
        epoch_val_val = 0.0
        epoch_val_acc = 0.0

        num_files_processed = 0

        for file_idx, file_path in enumerate(npz_files, start=1):
            print(f"\n[{file_idx}/{len(npz_files)}] Loading {os.path.basename(file_path)}")

            dataset = SingleNPZDataset(file_path)

            if len(dataset) < 2:
                print("  Skipping file: not enough samples.")
                continue

            train_size = int(TRAIN_VAL_SPLIT * len(dataset))
            val_size = len(dataset) - train_size

            if train_size == 0:
                train_size = len(dataset) - 1
                val_size = 1
            if val_size == 0:
                val_size = 1
                train_size = len(dataset) - 1

            train_dataset, val_dataset = random_split(
                dataset,
                [train_size, val_size],
                generator=torch.Generator().manual_seed(SEED)
            )

            train_loader = DataLoader(
                train_dataset,
                batch_size=BATCH_SIZE,
                shuffle=True,
                num_workers=NUM_WORKERS,
                pin_memory=(DEVICE.type == "cuda")
            )

            val_loader = DataLoader(
                val_dataset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=(DEVICE.type == "cuda")
            )

            train_loss, train_pol_loss, train_val_loss, train_acc = run_epoch(
                model, train_loader, optimizer, policy_criterion, value_criterion, train=True
            )

            val_loss, val_pol_loss, val_val_loss, val_acc = run_epoch(
                model, val_loader, optimizer, policy_criterion, value_criterion, train=False
            )

            print(f"  Train Loss    : {train_loss:.4f}")
            print(f"    Policy Loss : {train_pol_loss:.4f}")
            print(f"    Value Loss  : {train_val_loss:.4f}")
            print(f"    Policy Acc  : {train_acc:.4f}")

            print(f"  Val Loss      : {val_loss:.4f}")
            print(f"    Policy Loss : {val_pol_loss:.4f}")
            print(f"    Value Loss  : {val_val_loss:.4f}")
            print(f"    Policy Acc  : {val_acc:.4f}")

            epoch_train_loss += train_loss
            epoch_train_pol += train_pol_loss
            epoch_train_val += train_val_loss
            epoch_train_acc += train_acc

            epoch_val_loss += val_loss
            epoch_val_pol += val_pol_loss
            epoch_val_val += val_val_loss
            epoch_val_acc += val_acc

            num_files_processed += 1

            # Save rolling checkpoint after each chunk
            rolling_ckpt = os.path.join(SAVE_DIR, "last_model.pth")
            torch.save({
                "epoch": epoch,
                "file_idx": file_idx,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
            }, rolling_ckpt)

            # Save best checkpoint
            if val_loss < best_global_val_loss:
                best_global_val_loss = val_loss
                best_ckpt = os.path.join(SAVE_DIR, "best_model.pth")
                torch.save({
                    "file_idx": file_idx,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                }, best_ckpt)
                print("  Saved new best model.")

            del dataset, train_dataset, val_dataset, train_loader, val_loader
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        if num_files_processed == 0:
            print("No valid files processed this epoch.")
            continue

        epoch_train_loss /= num_files_processed
        epoch_train_pol /= num_files_processed
        epoch_train_val /= num_files_processed
        epoch_train_acc /= num_files_processed

        epoch_val_loss /= num_files_processed
        epoch_val_pol /= num_files_processed
        epoch_val_val /= num_files_processed
        epoch_val_acc /= num_files_processed

        print(f"\nEpoch {epoch} Summary:")
        print(f"  Avg Train Loss    : {epoch_train_loss:.4f}")
        print(f"  Avg Train PolLoss : {epoch_train_pol:.4f}")
        print(f"  Avg Train ValLoss : {epoch_train_val:.4f}")
        print(f"  Avg Train Acc     : {epoch_train_acc:.4f}")

        print(f"  Avg Val Loss      : {epoch_val_loss:.4f}")
        print(f"  Avg Val PolLoss   : {epoch_val_pol:.4f}")
        print(f"  Avg Val ValLoss   : {epoch_val_val:.4f}")
        print(f"  Avg Val Acc       : {epoch_val_acc:.4f}")

    print("\nRetraining finished.")


if __name__ == "__main__":
    main()

Found 3 npz files.
Loading checkpoint from: C:\Users\abdal\OneDrive\Desktop\chess-project\ver1.2\last_model.pth
Optimizer state loaded.
Resuming from epoch: 2
Previous best val loss: inf

Epoch 2/4

[1/3] Loading shard_curriculum_1.npz
  Train Loss    : 2.3250
    Policy Loss : 2.2517
    Value Loss  : 0.0733
    Policy Acc  : 0.4692
  Val Loss      : 2.3163
    Policy Loss : 2.2469
    Value Loss  : 0.0694
    Policy Acc  : 0.4695
  Saved new best model.

[2/3] Loading shard_curriculum_0.npz
  Train Loss    : 2.3293
    Policy Loss : 2.2558
    Value Loss  : 0.0735
    Policy Acc  : 0.4677
  Val Loss      : 2.3227
    Policy Loss : 2.2461
    Value Loss  : 0.0766
    Policy Acc  : 0.4711

[3/3] Loading shard_curriculum_2.npz
  Train Loss    : 2.3146
    Policy Loss : 2.2408
    Value Loss  : 0.0739
    Policy Acc  : 0.4721
  Val Loss      : 2.3065
    Policy Loss : 2.2355
    Value Loss  : 0.0711
    Policy Acc  : 0.4712
  Saved new best model.

Epoch 2 Summary:
  Avg Train Loss    : 

KeyboardInterrupt: 